# Compare Benchmark Runs
**Source:** Based on the example [compare-hypothesis.ipynb](https://github.com/Kotlin/kotlinx-benchmark/blob/master/examples/compare-hypothesis.ipynb) of the `kotlinx-benchmark` repository.

Grabs the last two runs and compares their score.

In [430]:
%use serialization, dataframe, kandy

In [431]:
@Serializable
public data class Benchmark(
    public val benchmark: String,
    public val mode: String,
    public val warmupIterations: Int,
    public val warmupTime: String,
    public val measurementIterations: Int,
    public val measurementTime: String,
    public val primaryMetric: PrimaryMetric,
    public val secondaryMetrics: Map<String, PrimaryMetric>,
    public val params: JsonObject? = null
)

@Serializable
public data class PrimaryMetric(
    public val score: Double,
    public val scoreError: Double,
    public val scoreConfidence: List<Double>,
    public val scorePercentiles: Map<String, Double>,
    public val scoreUnit: String,
    public val rawData: List<List<Double>>,
)

In [432]:
val nameLookupMap = mapOf(
    Pair("io.karpfen.features.BaseLineBenchmark.run", "Baseline"),
    Pair("io.karpfen.features.TogglePointOverheadBenchmark.run","Toggle-point"),
    Pair("io.karpfen.features.runtime.TickByTickOverhead.idle", "Tick-by-tick (idle)"),
    Pair("io.karpfen.features.runtime.BreakpointOverhead.idle", "Breakpoint (idle)"),
    Pair("io.karpfen.features.runtime.EventReplayOverhead.idle", "Event replay (idle)"),
    Pair("io.karpfen.features.runtime.EventReplayRecordingOverhead.record", "Event replay (recording)"),
    Pair("io.karpfen.features.runtime.EventReplayInjectionOverhead.inject", "Event replay (injection)"),
    Pair("io.karpfen.features.language.HistoryOverhead.idle", "History (idle)"),
    Pair("io.karpfen.features.language.TickLimitOverhead.idle", "Tick limit (idle)"),
    Pair("io.karpfen.features.language.TickLimitOverheadSpecializedNotSupported.notSupported", "Tick limit customized (workaround)"),
    Pair("io.karpfen.features.language.TickLimitOverheadSpecializedSupported.supported", "Tick limit customized (native)"),
    Pair("io.karpfen.features.language.HistoryOverheadSpecializedNotSupported.notSupported", "History customized (workaround)"),
    Pair("io.karpfen.features.language.HistoryOverheadSpecializedSupported.supported", "History customized (native)"),
)

In [433]:
//Fetch the last n benchmarks to include in the diagrams
val benchmarkCount = 2

In [434]:
import java.nio.file.Files
import java.nio.file.attribute.BasicFileAttributes
import kotlin.io.path.exists
import kotlin.io.path.forEachDirectoryEntry
import kotlin.io.path.isDirectory
import kotlin.io.path.listDirectoryEntries
import kotlin.io.path.readText

val runsDir = notebook.workingDir.resolve("../../../build/reports/benchmarks/main")
val outputFiles = runsDir.listDirectoryEntries()
    .filter { it.isDirectory() }
    .sortedByDescending { dir -> Files.readAttributes(dir, BasicFileAttributes::class.java).creationTime() }
    .subList(0, benchmarkCount)
    .map { it.resolve("benchmark.json") }

In [435]:
val json = Json { ignoreUnknownKeys = true }
val runs: List<List<Benchmark>> = outputFiles.map { file ->
    json.decodeFromString<List<Benchmark>>(file.readText())
}

In [436]:
import kotlinx.serialization.json.encodeToJsonElement
import org.jetbrains.letsPlot.core.plot.base.DataFrame

var combinedDf = emptyDataFrame<Benchmark>()
for (run in runs) {
    combinedDf = combinedDf.concat(run.toDataFrame() {
        "benchmark" from { nameLookupMap[it.benchmark] ?: it.benchmark }
        "score" from { it.primaryMetric.score }
        "scoreError" from { it.primaryMetric.scoreError }
        "deviationMin" from { it.primaryMetric.score - it.primaryMetric.scoreError }
        "deviationMax" from { it.primaryMetric.score + it.primaryMetric.scoreError }
    })
}
combinedDf = combinedDf.groupBy("benchmark").mean()
combinedDf

benchmark,score,scoreError,deviationMin,deviationMax
Tick limit customized (workaround),"3,946975","0,059992","3,886983","4,006967"
Tick limit customized (native),"2,405210","0,050416","2,354795","2,455626"


In [437]:
val minScore = combinedDf["deviationMin"].values().map { (it as Number).toDouble() }.minOrNull() ?: 0.0
val maxScore = combinedDf["deviationMax"].values().map { (it as Number).toDouble() }.maxOrNull() ?: 0.0

val lowerBound = minScore * 0.99
val upperBound = maxScore * 1.01

val plot = combinedDf.sortBy {"score".desc()}.plot {
    bars {
        x("benchmark")
        y("score") {
            scale = continuous(lowerBound..upperBound)
        }
    }
    errorBars {
        x("benchmark")
        yMin("deviationMin")
        yMax("deviationMax")
        width = 0.2
        borderLine.color = Color.BLACK
        borderLine.width = 0.75
    }
    coordinatesTransformation = CoordinatesTransformation.cartesianFlipped()
    layout {
        this.xAxisLabel = ""
        this.yAxisLabel = "ms/1000 ticks"
        style {
            global {
                title {
                    margin(10.0, -10.0)
                }
                text {
                    fontFamily = FontFamily.MONO
                }
            }
        }
        // Adjust the height of the Kandy plot based on the number of tests.
        size = 700 to ((50 * combinedDf.size().nrow) + 100)
    }
}
DISPLAY(HTML("<h4>Comparison</h4>"))
DISPLAY(plot)

Comparison

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="09cDCu" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 200.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("09cDCu");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"guides":{
"x":{
"title":""
},
"y":{
"title":"ms/1000 ticks"
}
},
"coord":{
"name":"flip",
"flip":true
},
"data":{
"score":[3.946974919632509,2.4052100651394897],
"deviationMax":[4.006967020035489,2.45562562182135],
"deviationMin":[3.886982819229528,2.3547945084576294],
"benchmark":["Tick limit customized (workaround)","Tick limit customized (native)"]
},
"ggsize":{
"width":700.0,
"height":200.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[2.3312465633730532,4.047036690235844]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"benchmark",
"y":"score"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"benchmark",
"ymin":"deviationMin",
"ymax":"deviationMax"
},
"stat":"identity",
"color":"#000000",
"size":0.75,
"sampling":"none",
"width":0.2,
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"text":{
"family":"mono",
"blank":false
},
"title":{
"margin":[10.0,-10.0,10.0,-10.0],
"blank":false
},
"axis_ontop":false,
"axis_ontop_y":false,
"axis_ontop_x":false
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"benchmark"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"deviationMin"
},{
"type":"float",
"column":"deviationMax"
}]
},
"spec_id":"272"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 2.5 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 3.5 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 Tick limit customized (workaround) 
 
 
 
 
 
 
 
 
 Tick limit customized (native) 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 ms/1000 ticks

In [438]:
val baseline = combinedDf["score"].values().map { (it as Number).toDouble() }.minOrNull()?: 0.0

var percentageDiffDf = combinedDf.mapToFrame {
    "benchmark" from { it.benchmark }
    "percentageDiff" from { (it.score - baseline) / baseline * 100 + 100}
    "percentageDeviationMin" from { (it.score - it.scoreError - baseline) / baseline * 100 + 100}
    "percentageDeviationMax" from { (it.score + it.scoreError - baseline) / baseline * 100 + 100}
}

percentageDiffDf

benchmark,percentageDiff,percentageDeviationMin,percentageDeviationMax
Tick limit customized (workaround),"164,101048","161,606792","166,595304"
Tick limit customized (native),"100,000000","97,903902","102,096098"


In [439]:
val minScore = percentageDiffDf["percentageDeviationMin"].values().map { (it as Number).toDouble() }.minOrNull() ?: 0.0
val maxScore = percentageDiffDf["percentageDeviationMax"].values().map { (it as Number).toDouble() }.maxOrNull() ?: 0.0

val lowerBound = minScore * 0.99
val upperBound = maxScore * 1.01

val plot = percentageDiffDf.sortBy {"percentageDiff".desc()}.plot {
    bars {
        x("benchmark")
        y("percentageDiff") {
            scale = continuous(lowerBound..upperBound)
            axis.breaks(format = "{.1f}%")
        }
    }
    errorBars {
        x("benchmark")
        yMin("percentageDeviationMin")
        yMax("percentageDeviationMax")
        width = 0.2
        borderLine.color = Color.BLACK
        borderLine.width = 0.75
    }
    coordinatesTransformation = CoordinatesTransformation.cartesianFlipped()
    layout {
        this.xAxisLabel = ""
        this.yAxisLabel = ""
        style {
            global {
                title {
                    margin(10.0, -10.0)
                }
                text {
                    fontFamily = FontFamily.MONO
                }
            }
        }
        // Adjust the height of the Kandy plot based on the number of tests.
        size = 700 to ((50 * combinedDf.size().nrow) + 100)
    }
}
DISPLAY(HTML("<h4>Comparison</h4>"))
DISPLAY(plot)

Comparison

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="GFTHkQ" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 200.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("GFTHkQ");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"guides":{
"x":{
"title":""
},
"y":{
"title":""
}
},
"coord":{
"name":"flip",
"flip":true
},
"data":{
"percentageDiff":[164.10104783939545,100.0],
"percentageDeviationMax":[166.59530400739055,102.09609785908394],
"percentageDeviationMin":[161.60679167140037,97.90390214091606],
"benchmark":["Tick limit customized (workaround)","Tick limit customized (native)"]
},
"ggsize":{
"width":700.0,
"height":200.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{.1f}%",
"limits":[96.9248631195069,168.26125704746445]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"benchmark",
"y":"percentageDiff"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"benchmark",
"ymin":"percentageDeviationMin",
"ymax":"percentageDeviationMax"
},
"stat":"identity",
"color":"#000000",
"size":0.75,
"sampling":"none",
"width":0.2,
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"text":{
"family":"mono",
"blank":false
},
"title":{
"margin":[10.0,-10.0,10.0,-10.0],
"blank":false
},
"axis_ontop":false,
"axis_ontop_y":false,
"axis_ontop_x":false
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"benchmark"
},{
"type":"float",
"column":"percentageDiff"
},{
"type":"float",
"column":"percentageDeviationMin"
},{
"type":"float",
"column":"percentageDeviationMax"
}]
},
"spec_id":"275"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 100.0% 
 
 
 
 
 
 
 110.0% 
 
 
 
 
 
 
 120.0% 
 
 
 
 
 
 
 130.0% 
 
 
 
 
 
 
 140.0% 
 
 
 
 
 
 
 150.0% 
 
 
 
 
 
 
 160.0% 
 
 
 
 
 
 
 170.0% 
 
 
 
 
 
 
 
 
 
 
 Tick limit customized (workaround) 
 
 
 
 
 
 
 
 
 Tick limit customized (native)